# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Prep/Setup

In [1]:
import os

REPO_URL = "https://github.com/tkg-create/FlyRank-ML-Track.git"
REPO_DIR = "FlyRank-ML-Track"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
assert os.path.isdir("work/notebooks"), "Not at repo root — check the clone worked"

Working dir: /content/FlyRank-ML-Track


In [2]:
%pip install -q tabulate

In [3]:
import duckdb
from getpass import getpass
import pandas as pd
import numpy as np

con = duckdb.connect()
hf_token = getpass("Paste your Hugging Face READ token: ")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"
month_path = f"{base}/fact_content_daily_performance/month=2026-03/data_0.parquet"

# Label — identical to w04/w05/w06
label_df = con.sql(f"""
    WITH halves AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date < '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
            SUM(CASE WHEN report_date >= '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_second_half
        FROM read_parquet('{month_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        content_hash_id,
        CASE WHEN impr_second_half < impr_first_half THEN 1 ELSE 0 END AS is_declining_proxy
    FROM halves
    WHERE impr_first_half > 0
""").df()

# Full-month position/click signal — for the baseline rule's score + tie-break only
pf = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_full,
        SUM(gsc_impressions) AS total_impressions_full,
        SUM(gsc_clicks) AS total_clicks_full
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
pf_valid = pf.dropna(subset=["avg_position_full"]).copy()
pf_valid["eligible"] = pf_valid["total_impressions_full"] >= 10
pf_valid["zero_clicks_at_position"] = (
    (pf_valid["avg_position_full"] <= 10) & (pf_valid["total_clicks_full"] == 0) & (pf_valid["eligible"])
).astype(int)

# First-half-only features — what the models actually train on
pf_fh = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_fh,
        SUM(gsc_impressions) AS total_impressions_fh,
        SUM(gsc_clicks) AS total_clicks_fh
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
pf_fh = pf_fh.dropna(subset=["avg_position_fh"]).copy()
pf_fh["ctr_fh"] = pf_fh["total_clicks_fh"] / pf_fh["total_impressions_fh"]

# Leakage-safe position trend, days 1-15 only
postrend = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN report_date < '2026-03-08' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk1,
        AVG(CASE WHEN report_date >= '2026-03-08' AND report_date < '2026-03-16' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_wk2
    FROM read_parquet('{month_path}')
    WHERE gsc_data_available IS TRUE AND report_date < '2026-03-16'
    GROUP BY content_hash_id
""").df()
postrend = postrend.dropna(subset=["avg_position_wk1", "avg_position_wk2"])
postrend["position_change"] = postrend["avg_position_wk2"] - postrend["avg_position_wk1"]
postrend["position_worsened"] = (postrend["position_change"] > 0).astype(int)
postrend_eligible = postrend.merge(pf_valid[["content_hash_id", "eligible"]], on="content_hash_id", how="left")
postrend_eligible = postrend_eligible[postrend_eligible["eligible"] == True].copy()

# Client map for grouping
client_map = con.sql(f"""
    SELECT DISTINCT content_hash_id, client_hash_id
    FROM read_parquet('{month_path}')
""").df()

# Assemble model_df — same joins, same order, as Week 5/6
model_df = pf_fh.merge(
    pf_valid[["content_hash_id", "total_impressions_full", "eligible", "zero_clicks_at_position"]],
    on="content_hash_id", how="inner"
)
model_df = model_df.merge(
    postrend_eligible[["content_hash_id", "position_change", "position_worsened"]],
    on="content_hash_id", how="left"
)
model_df["has_position_trend"] = model_df["position_change"].notna().astype(int)
model_df["position_change"] = model_df["position_change"].fillna(0)
model_df["position_worsened"] = model_df["position_worsened"].fillna(0).astype(int)
model_df = model_df.merge(client_map, on="content_hash_id", how="left").dropna(subset=["client_hash_id"])
model_df = model_df.merge(label_df, on="content_hash_id", how="inner").sort_values("content_hash_id").reset_index(drop=True)

model_df["log_impressions_fh"] = np.log1p(model_df["total_impressions_fh"])
model_df["log_clicks_fh"] = np.log1p(model_df["total_clicks_fh"])

feature_cols = ["avg_position_fh", "log_impressions_fh", "log_clicks_fh", "ctr_fh",
                "position_change", "has_position_trend"]

model_df["baseline_score"] = model_df["zero_clicks_at_position"] * 2 + model_df["position_worsened"]

print(f"model_df: {model_df.shape}, base rate: {model_df['is_declining_proxy'].mean():.3f}")

# --- Out-of-fold RF scoring: 5-fold GroupKFold, same seed as w05/w06 ---
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

def make_rf():
    return RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1)

X = model_df[feature_cols].astype(float)
y = model_df["is_declining_proxy"].astype(int)
groups = model_df["client_hash_id"]

oof_rf = np.full(len(model_df), np.nan)
gkf = GroupKFold(n_splits=5)
for fold_num, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
    rf = make_rf()
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_rf[test_idx] = rf.predict_proba(X.iloc[test_idx])[:, 1]

model_df["oof_rf_score"] = oof_rf
print(f"OOF coverage: {model_df['oof_rf_score'].notna().sum()} / {len(model_df)} rows")

Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

model_df: (150675, 16), base rate: 0.438
OOF coverage: 150675 / 150675 rows


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [4]:
# Thresholds — data-driven, not arbitrary
high_score_cut = model_df.loc[model_df["baseline_score"] == 0, "oof_rf_score"].quantile(0.90)
large_swing_cut = model_df["position_change"].abs().quantile(0.90)

def assign_archetype(row):
    if row["zero_clicks_at_position"] == 1 and row["position_worsened"] == 1:
        return 1, "zero_clicks_and_worsened", "Refresh content + overhaul title/meta"
    if row["zero_clicks_at_position"] == 1:
        return 2, "zero_clicks_only", "Overhaul title & meta"
    if row["position_worsened"] == 1:
        return 3, "position_worsened_only", "Refresh content"
    if row["baseline_score"] == 0 and row["oof_rf_score"] >= high_score_cut:
        caution = abs(row["position_change"]) >= large_swing_cut
        action = "Flag for manual review (model-only signal — verify before acting)" if caution \
                 else "Flag for manual review"
        return 4, "model_only_catch", action
    return 5, "no_flag", "Monitor"

archetype_info = model_df.apply(assign_archetype, axis=1, result_type="expand")
model_df[["priority_tier", "archetype", "action"]] = archetype_info

# Confidence scoring
low_data_cut = model_df["total_impressions_full"].quantile(0.25)
boundary_margin = 0.05 * high_score_cut  # within 5% of the model_only_catch cutoff

def assign_confidence(row):
    if row["total_impressions_full"] < low_data_cut:
        return "low"
    if row["archetype"] == "model_only_catch" and abs(row["oof_rf_score"] - high_score_cut) < boundary_margin:
        return "low"
    return "high"

model_df["confidence"] = model_df.apply(assign_confidence, axis=1)

# Combined score: model score does the ordering, rule adds a bounded nudge — neither signal fully vetoes the other
model_df["combined_score"] = model_df["oof_rf_score"] + 0.03 * model_df["baseline_score"]

ranked_queue = model_df.sort_values("combined_score", ascending=False).reset_index(drop=True)

print(ranked_queue["archetype"].value_counts())
print(ranked_queue[["content_hash_id", "archetype", "action", "baseline_score",
                     "oof_rf_score", "combined_score"]].head(20))
caution_count = (
    (ranked_queue["archetype"] == "model_only_catch") &
    (ranked_queue["position_change"].abs() >= large_swing_cut)
).sum()
print(f"\nmodel_only_catch rows with caution flag: {caution_count} / {(ranked_queue['archetype']=='model_only_catch').sum()}")

archetype
no_flag                     65567
position_worsened_only      52753
zero_clicks_only            14457
zero_clicks_and_worsened    10609
model_only_catch             7289
Name: count, dtype: int64
             content_hash_id               archetype                  action  \
0   content_403a36188e13fcc8  position_worsened_only         Refresh content   
1   content_2435b8bb25eeebd9  position_worsened_only         Refresh content   
2   content_df977de3b77ec57c  position_worsened_only         Refresh content   
3   content_b5a91be0a10cd899  position_worsened_only         Refresh content   
4   content_4e48bd81bb37eb4f  position_worsened_only         Refresh content   
5   content_5effb301ded55c21  position_worsened_only         Refresh content   
6   content_334bcb2761d0f9c7        model_only_catch  Flag for manual review   
7   content_83a700e06cf9e676        model_only_catch  Flag for manual review   
8   content_bab284527da6960a        model_only_catch  Flag for manual revi

The queue ranks all 150,675 scored pages by oof_rf_score (Week 5/6's random forest, five-fold GroupKFold, random_state=42) plus a small bonus from the Week 4 rule (0.03 × baseline_score, capped at 0.09), with the model doing nearly all the ordering and the rule only breaking close ties.

This replaces an earlier design where the rule's flags gated ranking outright — every row was sorted into its rule tier first, with the model only breaking ties inside that tier — which meant the model's evidence couldn't move a row out of a tier no matter how it scored. Week 6 found RF beats the rule at K=50 and up with a bootstrap CI clearly above zero, and a queue that lets a validated model do the ordering work is more consistent with that evidence than one where an untested Week 4 rule decides who's even in contention for the top of the list. Archetype and action come from the rule's flags alone, so reason and rank are answered separately, not by the same number.

There are five archetypes chosen, those being; zero_clicks_and_worsened (10,609 — refresh + overhaul), zero_clicks_only (14,457 — overhaul title/meta), position_worsened_only (52,753 — refresh), model_only_catch (7,289 — unflagged pages in the model's top 10%, sent to manual review), and no_flag (65,567 — monitor).

There is a caution flag this archetype was built to carry, that being the previously documented RF failure mode of high score driven by a large position swing rather than real decline. It was found to fire on only 42 of 7,289 rows (0.6%), marking it as a small minority needing a second look, not grounds to distrust the archetype.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This queue is built for a content editor or SEO strategist doing weekly or monthly triage — it tells them where to look first, not what to conclude. Every action is intended as a starting point for review, not an instruction to execute unread.

This whole system stops being valid past a few specific edges. The pipeline itself — feature construction, the GroupKFold fitting approach — is month-agnostic and can be rerun on any month's data, but the scores in this queue come from models trained only on March 2026. Nothing here was tested on another month, so a new month means retraining from scratch, not reapplying these scores to new pages.

The model was never given a direct zero-click signal (zero_clicks_at_position_fh was built but only used in a Week 6 diagnostic, never added to feature_cols), so it only picks up CTR problems indirectly. That's why zero-click archetypes carry the largest rule bonus in Section 1 but rarely reach the top of the blended queue — it's not that the rule is wrong, but rather the model isn't built to weigh that signal directly. Zero-click pages actually score respectably on average (0.481 vs. 0.423 for everything else), so the gap isn't that the model dismisses them — it's that the very top of the ranking is dominated by archetypes with a longer right tail of extreme scores, and zero-click pages rarely produce those extremes.

This queue can't detect a page that gets clicks but loses people immediately — the one candidate signal for that, session_rate, was tested in Week 4 and rejected because it moved in the wrong direction, confounded with impression volume rather than measuring real engagement. This is a gap in what the present signals measure, not something the model missed.

oof_rf_score is pooled from five separately trained fold models, and Week 6 found they aren't fully comparable — one fold's model produced systematically higher scores without being more accurate. It is best advised to treat close scores across the full population with some caution; a 0.02 gap between two rows isn't necessarily meaningful.

Confidence in the model's ordering is strong at K=50 and up, backed by a bootstrap CI clearly above zero, but soft at K=20, where RF only won 3 of 5 folds. Therefore the very top of the queue should be subject to more scrutiny than the model's confidence alone would suggest.

Finally, this queue recommends refresh candidates based on decline signals — it does not show or guarantee that refreshing causes recovery: acting on this queue and later seeing improvement doesn't directly confirm that the action caused it.

In [5]:
# Section 2 reference numbers — the concrete figures the limits paragraph above draws on
print(f"Scored population: {len(model_df):,} pages, single month (March 2026)")
print(f"Label window: is_declining_proxy compares impressions before vs after 2026-03-16")
print()

print("Features the model actually sees (feature_cols):")
print(feature_cols)
print(f"-> 'zero_clicks_at_position' is NOT in this list; the model only sees CTR indirectly via ctr_fh/log_clicks_fh")
print()

zero_click_archetypes = ["zero_clicks_and_worsened", "zero_clicks_only"]
zc_mean_score = model_df.loc[model_df["archetype"].isin(zero_click_archetypes), "oof_rf_score"].mean()
other_mean_score = model_df.loc[~model_df["archetype"].isin(zero_click_archetypes), "oof_rf_score"].mean()
print(f"Mean oof_rf_score, zero-click archetypes: {zc_mean_score:.3f}")
print(f"Mean oof_rf_score, everything else:        {other_mean_score:.3f}")
print()

print(f"Confidence thresholds in use:")
print(f"  low_data_cut (25th pct of total_impressions_full): {low_data_cut:.0f}")
print(f"  high_score_cut (90th pct oof_rf_score, unflagged):  {high_score_cut:.4f}")
print(f"  large_swing_cut (90th pct |position_change|):       {large_swing_cut:.2f}")
print()

print(f"Rows flagged low-confidence: {(model_df['confidence']=='low').sum():,} / {len(model_df):,} "
      f"({(model_df['confidence']=='low').mean():.1%})")

Scored population: 150,675 pages, single month (March 2026)
Label window: is_declining_proxy compares impressions before vs after 2026-03-16

Features the model actually sees (feature_cols):
['avg_position_fh', 'log_impressions_fh', 'log_clicks_fh', 'ctr_fh', 'position_change', 'has_position_trend']
-> 'zero_clicks_at_position' is NOT in this list; the model only sees CTR indirectly via ctr_fh/log_clicks_fh

Mean oof_rf_score, zero-click archetypes: 0.481
Mean oof_rf_score, everything else:        0.423

Confidence thresholds in use:
  low_data_cut (25th pct of total_impressions_full): 41
  high_score_cut (90th pct oof_rf_score, unflagged):  0.5182
  large_swing_cut (90th pct |position_change|):       12.00

Rows flagged low-confidence: 39,420 / 150,675 (26.2%)


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any row, check its confidence column. Rows marked low (39,420 of 150,675) sit on thin impression volume or, for model_only_catch specifically, too close to the score cutoff to trust the label on its own. This doesn't mean they're wrong per se, rather they're just under-evidenced and deserve a look at the actual page before the suggested action gets followed.

combined_score gaps under roughly 0.02–0.03 between neighboring rows shouldn't be read as one page clearly outranking another — oof_rf_score is pooled from five separately trained fold models that Week 6 found aren't fully comparable (one fold scored systematically higher without being more accurate), so small differences in a shared score can reflect which fold a row happened to land in rather than a real gap in risk.

Confidence in the model's ordering itself is stronger at K=50 and up than at K=20, where RF only won 3 of 5 folds against the rule. Treat the very top of the queue with a bit more scrutiny than its rank alone would suggest, especially early on.

If a client's pages look unusual against the rest of the population — very different traffic scale, a content type or vertical this pipeline hasn't seen much of — check that context before trusting the score. Week 6 found one client subset with a feature distribution sharply different from the rest, and that's exactly where the model performed worst and the baseline rule held up better.

Reviewer time isn't free, and archetypes don't pay off equally. `no_flag` (43.5% of the queue, lowest mean score at 0.401) isn't worth routine manual review — it's built for monitoring, not action. `model_only_catch` costs more per row (no rule backing it up) but pays off often, since only 0.6% trip the caution flag; `zero_clicks_and_worsened` is the best value in the queue — smallest group, highest mean score (0.494). Confidence is a cost signal too: low-confidence rows (26.2%) need more scrutiny per row than high-confidence ones. Working the queue in `combined_score` order already front-loads the best payoff-per-minute rows, which is a better use of limited review time than working archetype-by-archetype.

None of this should be automated past the review stage. No action — refresh, overhaul, anything — should be executed without a person reading the actual page first; the queue orders and reasons, it doesn't decide. No output from this queue should claim or imply to have predicted Google's algorithm, or that an action caused a later improvement, and nothing in this pipeline is a causal design. client_hash_id is for grouping only and should never become a model feature or appear in any client-facing export. Lastly, scores should never carry over between months — this queue is built on March 2026 data only, and a new month means retraining from scratch, not reapplying these numbers to new pages.

In [6]:
# Confidence Diagnostics
print(f"low_data_cut (impressions): {low_data_cut:.0f}")
print(f"boundary_margin (score): {boundary_margin:.4f}")
print()
print("Overall confidence split:")
print(model_df["confidence"].value_counts())
print()
print("Confidence by archetype:")
print(pd.crosstab(model_df["archetype"], model_df["confidence"]))
print()
print("Confidence within top 200 (the queue depth that matters most):")
print(ranked_queue.head(200)["confidence"].value_counts())

low_data_cut (impressions): 41
boundary_margin (score): 0.0259

Overall confidence split:
confidence
high    111255
low      39420
Name: count, dtype: int64

Confidence by archetype:
confidence                 high    low
archetype                             
model_only_catch           2561   4728
no_flag                   42824  22743
position_worsened_only    48348   4405
zero_clicks_and_worsened   7876   2733
zero_clicks_only           9646   4811

Confidence within top 200 (the queue depth that matters most):
confidence
high    200
Name: count, dtype: int64


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Some things can be checked the moment a new month's queue is built, before anyone acts on it. If the archetype mix drifts far from this month's shape — `no_flag` swelling past its current 43.5%, or `model_only_catch` moving well off its designed ~10% share of the unflagged population — something changed upstream before a single score can be trusted. The same goes for the underlying feature distributions (`avg_position_fh`, `ctr_fh`, `log_impressions_fh`): a large shift from this month's values is the same warning sign Week 6 found in its one unusual client subset, just caught proactively instead of after the fact.

Section 1's rule-agreement check is worth rerunning too — if the model's mean score stops rising in the rule's own severity order across the three rule-only archetypes, something structural has broken and it's not just noise. And a basic schema check — the warehouse columns this pipeline reads still existing, still populated — costs nothing and catches the most boring failure mode before it becomes a confusing one.

Everything else can only be confirmed once a later month's real outcomes exist. The one that matters most is precision@K on the next period's actual declines, compared against the Week 5/6 benchmarks — an average precision drop for the whole queue is the headline signal that the model is no longer predicting anything useful.

A more specific version of the same check is running it per archetype rather than in aggregate: one reason code going stale (say, `position_worsened_only` stops predicting anything) is a different, more actionable finding than an overall dip, and would point at what to fix rather than just that something's wrong. The `model_only_catch` caution flag is worth revisiting too — if the rows it caught turn out to be false positives at a meaningfully higher rate than the rest of the archetype, that would say the large-swing heuristic itself needs revisiting, not just the rows it's currently catching.

None of these are automatic retrain triggers on their own — they're just good checks to make before deciding whether retraining or a closer audit is warranted. Given the model was only ever validated on a single month, the more conservative default is retraining every time this queue is rebuilt for a new month, rather than waiting for one of these signals to fire.

In [7]:
# Baseline reference numbers — what a future month's monitoring checks would compare against

archetype_pct = model_df["archetype"].value_counts(normalize=True).sort_values(ascending=False)
print("Current archetype mix (compare future months against this):")
print(archetype_pct.apply(lambda x: f"{x:.1%}"))
print()

rule_only = ["zero_clicks_and_worsened", "zero_clicks_only", "position_worsened_only", "no_flag"]
rule_agreement = model_df.loc[model_df["archetype"].isin(rule_only)].groupby("archetype")["oof_rf_score"].mean().sort_values(ascending=False)
print("Rule-agreement check baseline (should stay in this order):")
print(rule_agreement.round(3))
print()

print("Feature distribution baseline (compare future months' summary stats against these):")
print(model_df[feature_cols].describe().round(3))
print()

print(f"model_only_catch caution-flag rate this month: {caution_count} / "
      f"{(model_df['archetype']=='model_only_catch').sum()} "
      f"({caution_count / (model_df['archetype']=='model_only_catch').sum():.1%})")

unflagged_pct_of_unflagged = (model_df["archetype"] == "model_only_catch").sum() / (model_df["baseline_score"] == 0).sum()
print(f"model_only_catch as % of unflagged population (the number to track): {unflagged_pct_of_unflagged:.1%}")

Current archetype mix (compare future months against this):
archetype
no_flag                     43.5%
position_worsened_only      35.0%
zero_clicks_only             9.6%
zero_clicks_and_worsened     7.0%
model_only_catch             4.8%
Name: proportion, dtype: object

Rule-agreement check baseline (should stay in this order):
archetype
zero_clicks_and_worsened    0.494
zero_clicks_only            0.472
position_worsened_only      0.432
no_flag                     0.401
Name: oof_rf_score, dtype: float64

Feature distribution baseline (compare future months' summary stats against these):
       avg_position_fh  log_impressions_fh  log_clicks_fh      ctr_fh  \
count       150675.000          150675.000     150675.000  150675.000   
mean            16.547               4.630          0.527       0.004   
std             18.215               2.288          0.902       0.034   
min              0.041               0.693          0.000       0.000   
25%              5.187               

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
import os
import json
import matplotlib.pyplot as plt

OUTPUT_DIR = "work/outputs"
CHART_DIR = f"{OUTPUT_DIR}/charts"
os.makedirs(CHART_DIR, exist_ok=True)

# --- 1. Metrics JSON — the receipts every number in this notebook traces back to ---
archetype_counts = ranked_queue["archetype"].value_counts().to_dict()
rule_agreement_means = (
    model_df.loc[model_df["archetype"].isin(
        ["zero_clicks_and_worsened", "zero_clicks_only", "position_worsened_only", "no_flag"]
    )].groupby("archetype")["oof_rf_score"].mean().round(3).to_dict()
)
confidence_counts = model_df["confidence"].value_counts().to_dict()

metrics = {
    "population_size": len(model_df),
    "label_window": "impressions before vs after 2026-03-16, month=2026-03",
    "seed": 42,
    "thresholds": {
        "high_score_cut": round(float(high_score_cut), 4),
        "large_swing_cut": round(float(large_swing_cut), 2),
        "low_data_cut": round(float(low_data_cut), 1),
        "boundary_margin": round(float(boundary_margin), 4),
    },
    "archetype_counts": archetype_counts,
    "rule_agreement_mean_scores": rule_agreement_means,
    "confidence_split": confidence_counts,
    "confidence_low_pct": round((model_df["confidence"] == "low").mean(), 3),
    "model_only_catch_caution": {
        "flagged": int(caution_count),
        "total": int((model_df["archetype"] == "model_only_catch").sum()),
        "rate": round(caution_count / (model_df["archetype"] == "model_only_catch").sum(), 4),
    },
    "feature_distribution_baseline": model_df[feature_cols].describe().round(3).to_dict(),
}

with open(f"{OUTPUT_DIR}/w07_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote {OUTPUT_DIR}/w07_metrics.json")

# --- 2. Charts ---
fig, ax = plt.subplots(figsize=(8, 4))
counts = ranked_queue["archetype"].value_counts().sort_values()
ax.barh(counts.index, counts.values, color="#426B69")
ax.set_title("Archetype mix")
ax.set_xlabel("Rows")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/archetype_mix.svg")
plt.close()

fig, ax = plt.subplots(figsize=(6, 4))
conf_counts = model_df["confidence"].value_counts().reindex(["high", "low"])
ax.bar(conf_counts.index, conf_counts.values, color="#6F4E7C")
ax.set_title("Confidence split")
ax.set_ylabel("Rows")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/confidence_mix.svg")
plt.close()

print(f"Wrote charts to {CHART_DIR}/")

# --- 3. Markdown report — mirrors the reference's model_report.md shape ---
top20 = ranked_queue.head(20)[["content_hash_id", "archetype", "action", "confidence", "combined_score"]]

report = f"""# ML-10 Content Action Playbook Report

Scored population: {len(model_df):,} pages, March 2026, `is_declining_proxy` label.

## Archetype breakdown

| Archetype | Count | Action |
|---|---:|---|
"""
action_by_archetype = ranked_queue.groupby("archetype")["action"].first()
for archetype, count in ranked_queue["archetype"].value_counts().items():
    report += f"| `{archetype}` | {count:,} | {action_by_archetype[archetype]} |\n"

report += f"""
## Confidence split

- High: {confidence_counts.get('high', 0):,}
- Low: {confidence_counts.get('low', 0):,} ({(model_df['confidence']=='low').mean():.1%})

## Rule-agreement check (model score vs. rule severity, rule-only archetypes)

| Archetype | Mean oof_rf_score |
|---|---:|
"""
for archetype, score in rule_agreement_means.items():
    report += f"| `{archetype}` | {score:.3f} |\n"

report += f"""
## model_only_catch caution flag

{caution_count} of {(model_df['archetype']=='model_only_catch').sum()} rows ({caution_count/(model_df['archetype']=='model_only_catch').sum():.1%}) flagged for large position-swing caution.

## Top 20 queue preview

{top20.to_markdown(index=False)}

## Practical use

Use this queue as a reviewer aid, not an automatic action trigger. See Section 2 (intended use and limits) and Section 3 (human review and no-go list) in the notebook for the full detail — every action here is a starting point for review, not an instruction to execute unread.

## Generated files

- `work/outputs/w07_metrics.json`
- `work/outputs/charts/archetype_mix.svg`
- `work/outputs/charts/confidence_mix.svg`
- `work/outputs/w07_ranked_queue.csv` (local only — gitignored, not committed; rebuild from this notebook's Sections 0–1 rather than expecting this file to exist in a fresh clone)
"""

with open(f"{OUTPUT_DIR}/w07_report.md", "w") as f:
    f.write(report)
print(f"Wrote {OUTPUT_DIR}/w07_report.md")

# --- 4. Full queue CSV — local convenience only, never committed ---
ranked_queue.to_csv(f"{OUTPUT_DIR}/w07_ranked_queue.csv", index=False)
print(f"Wrote {OUTPUT_DIR}/w07_ranked_queue.csv (gitignored — local only)")

Wrote work/outputs/w07_metrics.json
Wrote charts to work/outputs/charts/
Wrote work/outputs/w07_report.md
Wrote work/outputs/w07_ranked_queue.csv (gitignored — local only)


In [9]:
from getpass import getpass

github_token = getpass("Paste your GitHub PAT (repo scope): ")

!git config user.email "tkg-create@users.noreply.github.com"
!git config user.name "tkg-create"

!git pull origin main
!git add work/outputs
!git commit -m "w07: action playbook exports (metrics json, charts, report)"
!git remote set-url origin https://{github_token}@github.com/tkg-create/FlyRank-ML-Track.git
!git push origin main

Paste your GitHub PAT (repo scope): ··········
From https://github.com/tkg-create/FlyRank-ML-Track
 * branch            main       -> FETCH_HEAD
Already up to date.
[main 40a2e22] w07: action playbook exports (metrics json, charts, report)
 4 files changed, 2027 insertions(+)
 create mode 100644 work/outputs/charts/archetype_mix.svg
 create mode 100644 work/outputs/charts/confidence_mix.svg
 create mode 100644 work/outputs/w07_metrics.json
 create mode 100644 work/outputs/w07_report.md
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 10.61 KiB | 3.54 MiB/s, done.
Total 9 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 2 local objects.
To https://github.com/tkg-create/FlyRank-ML-Track.git
   c5c1612..40a2e22  main -> main


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.